In [1]:
import pandas as pd

In [ ]:
# read source data
tmp = pd.read_csv(
  'source_data/아파트(매매)_실거래가/아파트(매매)_실거래가_20260412173112.csv',
  # encoding='cp949',
  # dtype='string',
  # na_values='-',
  # skiprows=15
)
tmp

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,1,경기도 평택시 소사동,697,0697,0000,평택효성해링턴플레이스2단지,72.7680,202101,31,"31,000",<NA>,13,<NA>,<NA>,2019,소사3로 22,<NA>,<NA>,<NA>,<NA>
1,2,경기도 평택시 동삭동,854,0854,0000,평택센트럴자이 4단지,75.7115,202101,31,"49,000",<NA>,6,<NA>,<NA>,2018,상서재로 50,<NA>,<NA>,<NA>,<NA>
2,3,울산광역시 울주군 온양읍 망양리,가-5-1,0005,0001,회야리버,45.0730,202101,31,"7,500",<NA>,6,<NA>,<NA>,2000,원동2길 19-1,<NA>,<NA>,<NA>,<NA>
3,4,경상남도 김해시 부원동,601-22,0601,0022,편한세상,39.4700,202101,31,"10,200",<NA>,8,<NA>,<NA>,2016,가락로29번길 10,20220129,<NA>,<NA>,<NA>
4,5,전북특별자치도 부안군 부안읍 선은리,258-1,0258,0001,대림낭주골,59.9250,202101,31,"8,200",<NA>,17,<NA>,<NA>,1997,낭주길 5,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62546,62547,경기도 화성시 동탄구 청계동,512,0512,0000,동탄역 시범우남퍼스트빌아파트,73.9600,202101,01,"99,500",<NA>,8,<NA>,<NA>,2015,동탄대로시범길 276,<NA>,<NA>,<NA>,<NA>
62547,62548,서울특별시 구로구 신도림동,640,0640,0000,신성은하수,59.8800,202101,01,"77,500",<NA>,11,<NA>,<NA>,1997,경인로66길 5,<NA>,<NA>,<NA>,<NA>
62548,62549,충청북도 제천시 청전동,4,0004,0000,주공,41.9800,202101,01,"1,600",<NA>,5,<NA>,<NA>,1980,청전대로15길 70,<NA>,<NA>,<NA>,<NA>
62549,62550,경기도 화성시 효행구 기안동,895,0895,0000,기안마을풍성신미주,84.8008,202101,01,"20,800",<NA>,1,<NA>,<NA>,2004,융건로 99,<NA>,<NA>,<NA>,<NA>


In [71]:
tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62551 entries, 0 to 62550
Data columns (total 20 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   NO        62551 non-null  string
 1   시군구       62551 non-null  string
 2   번지        62551 non-null  string
 3   본번        62551 non-null  string
 4   부번        62551 non-null  string
 5   단지명       62551 non-null  string
 6   전용면적(㎡)   62551 non-null  string
 7   계약년월      62551 non-null  string
 8   계약일       62551 non-null  string
 9   거래금액(만원)  62551 non-null  string
 10  동         62551 non-null  string
 11  층         62551 non-null  string
 12  매수자       62551 non-null  string
 13  매도자       62551 non-null  string
 14  건축년도      62551 non-null  string
 15  도로명       62551 non-null  string
 16  해제사유발생일   62551 non-null  string
 17  거래유형      62551 non-null  string
 18  중개사소재지    62551 non-null  string
 19  등기일자      62551 non-null  string
dtypes: string(20)
memory usage: 9.5 MB


In [82]:
# 수도권/비수도권 분류
tmp['sido_nm'] = [
  sgg.split(' ')[0]
  for sgg
  in tmp['시군구']
]

tmp['is_sdg'] = [
  sido_nm in ('서울특별시','인천광역시','경기도')
  for sido_nm
  in tmp['sido_nm']
]

In [83]:
print(
  '수도권 목록:\n',
  tmp[tmp['is_sdg'] == True]['sido_nm'].value_counts()
)
print(
  '비수도권 목록:\n',
  tmp[tmp['is_sdg'] == False]['sido_nm'].value_counts()
)

수도권 목록:
 sido_nm
경기도      20308
서울특별시     5956
인천광역시     4761
Name: count, dtype: int64
비수도권 목록:
 sido_nm
경상남도       4362
경상북도       3476
충청남도       3159
충청북도       3038
부산광역시      3036
강원특별자치도    2521
전북특별자치도    2224
대구광역시      2147
대전광역시      1923
광주광역시      1861
전라남도       1668
울산광역시      1248
세종특별자치시     455
제주특별자치도     408
Name: count, dtype: int64


In [84]:
# 계약년월
tmp['contract_dt'] = pd.to_datetime(tmp['계약년월']+[dt.zfill(2) for dt in tmp['계약일']])

In [85]:
print(tmp['contract_dt'])

0       2021-01-31
1       2021-01-31
2       2021-01-31
3       2021-01-31
4       2021-01-31
           ...    
62546   2021-01-01
62547   2021-01-01
62548   2021-01-01
62549   2021-01-01
62550   2021-01-01
Name: contract_dt, Length: 62551, dtype: datetime64[ns]


In [91]:
# 유효하지 않은 데이터 제거
tmp[tmp['해제사유발생일'].notna()]

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,sido_nm,is_sdg,contract_dt
3,4,경상남도 김해시 부원동,601-22,0601,0022,편한세상,39.4700,202101,31,"10,200",...,<NA>,2016,가락로29번길 10,20220129,<NA>,<NA>,<NA>,경상남도,False,2021-01-31
23,24,전북특별자치도 군산시 지곡동,607,0607,0000,쌍용예가,84.9857,202101,31,"29,900",...,<NA>,2014,계산로 71,20210217,<NA>,<NA>,<NA>,전북특별자치도,False,2021-01-31
35,36,전북특별자치도 전주시 덕진구 진북동,416,0416,0000,우성,134.3400,202101,31,"21,500",...,<NA>,1995,태진로 101,20210204,<NA>,<NA>,<NA>,전북특별자치도,False,2021-01-31
54,55,강원특별자치도 태백시 황지동,846,0846,0000,대산2차하이츠빌,84.8910,202101,31,"21,500",...,<NA>,2014,번영로 289,20210301,<NA>,<NA>,<NA>,강원특별자치도,False,2021-01-31
73,74,강원특별자치도 원주시 관설동,1713-2,1713,0002,청솔8차,59.8500,202101,31,"9,600",...,<NA>,2002,나비허리길 188,20210212,<NA>,<NA>,<NA>,강원특별자치도,False,2021-01-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62473,62474,대구광역시 수성구 매호동,721-1,0721,0001,매호청구하이츠빌,84.9950,202101,01,"29,100",...,<NA>,2000,천을로23길 26,20210330,<NA>,<NA>,<NA>,대구광역시,False,2021-01-01
62481,62482,부산광역시 사상구 주례동,530-5,0530,0005,현대무지개타운,105.2400,202101,01,"22,800",...,<NA>,1993,주례로 93,20210126,<NA>,<NA>,<NA>,부산광역시,False,2021-01-01
62489,62490,부산광역시 남구 용호동,776-12,0776,0012,롯데캐슬아인스,59.9329,202101,01,"22,800",...,<NA>,2007,용호로231번길 62,20210202,<NA>,<NA>,<NA>,부산광역시,False,2021-01-01
62507,62508,서울특별시 영등포구 대림동,1125,1125,0000,보라매 신동아 파밀리에,84.8800,202101,01,"100,000",...,<NA>,2016,디지털로 420,20210204,<NA>,<NA>,<NA>,서울특별시,True,2021-01-01
